# DDoS Detection System - Exploratory Analysis

This notebook provides exploratory analysis of the DDoS detection system:
- Traffic pattern visualization
- Feature distribution analysis
- Model performance evaluation
- Attack type comparison

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from data.generate_dataset import TrafficGenerator
from data.feature_extraction import TrafficFeatures

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 8)

%matplotlib inline

## 1. Generate Sample Traffic Patterns

In [ ]:
# Generate samples for each pattern
patterns = ['normal', 'syn_flood', 'udp_flood', 'http_flood', 'dns_amplification']
samples_per_pattern = 1000

data = {pattern: [] for pattern in patterns}

for pattern in patterns:
    for _ in range(samples_per_pattern):
        sample = TrafficGenerator.generate_sample(
            TrafficGenerator.PATTERNS[pattern],
            add_noise=True
        )
        data[pattern].append(sample)

# Convert to DataFrames
dfs = {}
feature_names = TrafficFeatures.get_feature_names()

for pattern in patterns:
    dfs[pattern] = pd.DataFrame(data[pattern], columns=feature_names)
    dfs[pattern]['pattern'] = pattern

# Combine all patterns
df_all = pd.concat(dfs.values(), ignore_index=True)
print(f"Generated {len(df_all)} samples across {len(patterns)} patterns")

## 2. Feature Distribution Analysis

In [ ]:
# Plot feature distributions
fig, axes = plt.subplots(4, 2, figsize=(15, 16))
axes = axes.ravel()

for idx, feature in enumerate(feature_names):
    for pattern in patterns:
        axes[idx].hist(
            dfs[pattern][feature],
            bins=50,
            alpha=0.5,
            label=pattern
        )
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'Distribution of {feature}')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Feature Correlation Analysis

In [ ]:
# Compute correlation matrix for each pattern
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for idx, pattern in enumerate(patterns):
    corr = dfs[pattern][feature_names].corr()
    sns.heatmap(
        corr,
        annot=True,
        fmt='.2f',
        cmap='coolwarm',
        center=0,
        ax=axes[idx],
        square=True
    )
    axes[idx].set_title(f'{pattern.upper()} - Feature Correlation')

# Remove extra subplot
fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

## 4. Pattern Comparison - Box Plots

In [ ]:
# Box plots for key features
key_features = ['packet_rate', 'byte_rate', 'syn_ratio', 'protocol_entropy']

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for idx, feature in enumerate(key_features):
    df_all.boxplot(
        column=feature,
        by='pattern',
        ax=axes[idx]
    )
    axes[idx].set_title(f'{feature} by Pattern')
    axes[idx].set_xlabel('Pattern')
    axes[idx].set_ylabel(feature)
    plt.sca(axes[idx])
    plt.xticks(rotation=45)

plt.suptitle('Feature Comparison Across Patterns', y=1.02)
plt.tight_layout()
plt.show()

## 5. Time Series Visualization

In [ ]:
# Generate time series for visualization
sequence_length = 100

sequences = {}
for pattern in patterns:
    seq = TrafficGenerator.generate_sequence(pattern, sequence_length)
    sequences[pattern] = seq

# Plot time series
fig, axes = plt.subplots(4, 1, figsize=(15, 16))

plot_features = ['packet_rate', 'byte_rate', 'syn_ratio', 'protocol_entropy']

for idx, feature in enumerate(plot_features):
    feature_idx = feature_names.index(feature)
    
    for pattern in patterns:
        axes[idx].plot(
            sequences[pattern][:, feature_idx],
            label=pattern,
            alpha=0.7
        )
    
    axes[idx].set_xlabel('Time Step')
    axes[idx].set_ylabel(feature)
    axes[idx].set_title(f'{feature} Over Time')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Transition Sequence Analysis

In [ ]:
# Generate transition sequence
transition_seq, transition_labels = TrafficGenerator.generate_transition_sequence(
    'normal',
    'syn_flood',
    sequence_length=100,
    transition_point=50
)

# Plot transition
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# Plot features
axes[0].plot(transition_seq[:, 0], label='Packet Rate')
axes[0].plot(transition_seq[:, 4] * 1000, label='SYN Ratio (×1000)')
axes[0].axvline(x=50, color='r', linestyle='--', label='Transition Point')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Value')
axes[0].set_title('Traffic Features During Normal → Attack Transition')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot labels
axes[1].plot(transition_labels, color='red', linewidth=2)
axes[1].fill_between(range(len(transition_labels)), transition_labels, alpha=0.3)
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Label')
axes[1].set_title('Ground Truth Labels')
axes[1].set_ylim([-0.1, 1.1])
axes[1].grid(True, alpha=0.3)

# Plot unique source IPs
axes[2].plot(transition_seq[:, 5], color='purple')
axes[2].axvline(x=50, color='r', linestyle='--', label='Transition Point')
axes[2].set_xlabel('Time Step')
axes[2].set_ylabel('Count')
axes[2].set_title('Unique Source IPs Over Time')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Statistical Summary

In [ ]:
# Generate summary statistics for each pattern
for pattern in patterns:
    print(f"\n{'='*80}")
    print(f"{pattern.upper()} PATTERN STATISTICS")
    print('='*80)
    print(dfs[pattern][feature_names].describe())

## 8. Model Performance Analysis (After Training)

In [ ]:
# This section requires a trained model
# Uncomment and run after training

# from models.ssm_model import DDoSDetector
# from data.feature_extraction import FeatureNormalizer
#
# # Load model
# model = DDoSDetector()
# checkpoint = torch.load('../checkpoints/best_model.pt')
# model.load_state_dict(checkpoint['model_state_dict'])
# model.eval()
#
# # Load normalizer
# normalizer = FeatureNormalizer(feature_dim=8)
# normalizer.load('../checkpoints/normalizer.npz')
#
# # Generate test sequences
# test_sequences = []
# test_labels = []
# for pattern in patterns:
#     for _ in range(20):
#         seq = TrafficGenerator.generate_sequence(pattern, 60)
#         test_sequences.append(normalizer.normalize(seq))
#         test_labels.append(0 if pattern == 'normal' else 1)
#
# X_test = torch.FloatTensor(np.array(test_sequences))
#
# # Predict
# with torch.no_grad():
#     predictions = model.predict(X_test)
#
# # Compute metrics
# from sklearn.metrics import confusion_matrix, classification_report
# y_pred = predictions['is_attack'].numpy()
# y_true = np.array(test_labels)
#
# print(classification_report(y_true, y_pred))
# print("\nConfusion Matrix:")
# print(confusion_matrix(y_true, y_pred))

## Conclusion

This notebook demonstrates:
- Clear separation between normal and attack traffic patterns
- Distinct feature distributions for different attack types
- Smooth transitions in simulated attack scenarios
- Feature correlations vary by traffic pattern

These insights validate the feasibility of using temporal models for DDoS detection.